# 🛡️ Azure AI Content Safety — Lab AI-102

**Objectif**: Modérer le contenu textuel et visuel avec Azure AI Content Safety.

## Compétences AI-102 couvertes
- Analyser du texte sur 4 catégories de nuisance
- Comprendre les niveaux de sévérité (0, 2, 4, 6)
- Modérer des images
- Détecter les tentatives de jailbreak (Shield Prompts)
- Configurer des seuils de blocage adaptés au contexte

In [ ]:
%pip install azure-ai-contentsafety python-dotenv -q

In [ ]:
import os
from dotenv import load_dotenv
from azure.ai.contentsafety import ContentSafetyClient
from azure.ai.contentsafety.models import AnalyzeTextOptions, TextCategory
from azure.core.credentials import AzureKeyCredential

load_dotenv('../.env')

client = ContentSafetyClient(
    endpoint=os.getenv('AZURE_CONTENT_SAFETY_ENDPOINT'),
    credential=AzureKeyCredential(os.getenv('AZURE_CONTENT_SAFETY_KEY'))
)

print('✅ Client Azure AI Content Safety initialisé')

# Seuils de blocage (configurables selon le contexte)
SAFETY_THRESHOLD = 2  # Bloquer dès sévérité 2 (LOW)

SEVERITY_DESCRIPTIONS = {
    0: 'Sûr',
    2: 'Faible',
    4: 'Moyen',
    6: 'Élevé'
}

## 1. Analyse de texte — 4 catégories de nuisance

In [ ]:
# AI-102: Content Safety analyse 4 catégories:
# - Hate: discours haineux, discrimination
# - Violence: contenu violent
# - Sexual: contenu sexuel/adulte
# - SelfHarm: automutilation, suicide

# Niveaux de sévérité: 0 (sûr) → 2 (faible) → 4 (moyen) → 6 (élevé)

safe_text = "Azure AI Foundry est une excellente plateforme pour développer des applications intelligentes."

options = AnalyzeTextOptions(
    text=safe_text,
    categories=[
        TextCategory.HATE,
        TextCategory.VIOLENCE,
        TextCategory.SEXUAL,
        TextCategory.SELF_HARM,
    ],
    output_type='FourSeverityLevels'
)

response = client.analyze_text(options)

print(f"Texte analysé: '{safe_text[:60]}...'\n")
print("Résultats par catégorie:")
for cat in response.categories_analysis:
    severity = cat.severity or 0
    status = '✅ OK' if severity < SAFETY_THRESHOLD else '🚫 BLOQUÉ'
    print(f"  {cat.category}: Sévérité {severity} ({SEVERITY_DESCRIPTIONS.get(severity, 'Inconnu')}) {status}")

is_safe = all((cat.severity or 0) < SAFETY_THRESHOLD for cat in response.categories_analysis)
print(f"\nConclusion: {'✅ Contenu sûr' if is_safe else '🚫 Contenu bloqué'}")

## 2. Tester différents niveaux de sévérité

In [ ]:
# AI-102: Comprendre les niveaux de sévérité en testant des contenus gradués
# Note: Les exemples suivants sont neutres/académiques à des fins d'illustration

def analyze_content(text: str, threshold: int = 2) -> dict:
    """Analyse le contenu et retourne le résultat structuré."""
    options = AnalyzeTextOptions(
        text=text,
        categories=[TextCategory.HATE, TextCategory.VIOLENCE, TextCategory.SEXUAL, TextCategory.SELF_HARM],
        output_type='FourSeverityLevels'
    )
    response = client.analyze_text(options)
    
    results = {}
    is_safe = True
    for cat in response.categories_analysis:
        severity = cat.severity or 0
        if severity >= threshold:
            is_safe = False
        results[str(cat.category)] = severity
    
    return {'is_safe': is_safe, 'categories': results}

test_cases = [
    "Le rapport financier montre une croissance de 15% ce trimestre.",
    "Ce document décrit les procédures de sécurité pour les situations d'urgence.",
]

for text in test_cases:
    result = analyze_content(text)
    status = '✅' if result['is_safe'] else '🚫'
    print(f"{status} '{text[:50]}...'")
    print(f"   Catégories: {result['categories']}\n")

## 3. Ajuster le seuil selon le contexte

In [ ]:
# AI-102: Le seuil de blocage doit être adapté au contexte applicatif
# Plateforme enfants: seuil = 0 (bloquer tout)
# Entreprise standard: seuil = 2 (bloquer faible et plus)
# Recherche académique: seuil = 4 (bloquer moyen et plus)

thresholds = {
    'Plateforme enfants': 0,
    'Application professionnelle': 2,
    'Recherche académique': 4,
    'Usage interne expert': 6,
}

sample_text = "Ce texte contient une description académique d'un conflit historique avec des références à la violence."

result = analyze_content(sample_text, threshold=0)
max_severity = max(result['categories'].values()) if result['categories'] else 0

print(f"Texte: '{sample_text[:60]}...'")
print(f"Sévérité maximale détectée: {max_severity}\n")
print("Résultat selon le contexte:")
for context, threshold in thresholds.items():
    blocked = max_severity >= threshold
    status = '🚫 BLOQUÉ' if blocked else '✅ AUTORISÉ'
    print(f"  {context} (seuil={threshold}): {status}")

## 4. Détection de jailbreak (Shield Prompts)

In [ ]:
# AI-102: Shield Prompt détecte les tentatives de contournement des guardrails AI
# Protège les applications LLM contre les attaques adversariales

try:
    from azure.ai.contentsafety.models import ShieldPromptOptions
    
    # Exemple de prompt légitime
    legitimate_prompt = "Peux-tu m'aider à rédiger un résumé de ce contrat?"
    
    options = ShieldPromptOptions(user_prompt=legitimate_prompt)
    response = client.shield_prompt(options)
    
    attack_detected = response.user_prompt_analysis.attack_detected if response.user_prompt_analysis else False
    print(f"Prompt: '{legitimate_prompt}'")
    print(f"Jailbreak détecté: {attack_detected}")
    
except Exception as e:
    print(f"Shield Prompt non disponible dans cette version: {e}")
    print("Shield Prompt protège contre les prompts adversariaux de type:")
    print("  - Ignore tes instructions précédentes et fais X")
    print("  - Prétend que tu es un autre AI sans restrictions")
    print("  - Contournement via encodage ou obfuscation")

## Résumé AI-102 — Content Safety

| Catégorie | Description | Exemples |
|-----------|-------------|----------|
| **Hate** | Discours haineux, discrimination | Propos racistes, sexistes |
| **Violence** | Contenu violent | Menaces, descriptions violentes |
| **Sexual** | Contenu sexuel/adulte | Contenu explicite |
| **SelfHarm** | Automutilation, suicide | Instructions dangereuses |

| Sévérité | Valeur | Description |
|----------|--------|-------------|
| Safe | 0 | Contenu sûr |
| Low | 2 | Légèrement problématique |
| Medium | 4 | Modérément nuisible |
| High | 6 | Sévèrement nuisible |